# About: NFSの準備

CoursewareHub用のNFSを準備する

* NFSサーバーの設定
  * NFS serverのインストール
  * CoursewareHubのノードへのexportを設定
* CousewareHubのノードにNFSマウントを設定
  * NFS clientのインストール
  * fstabの設定

# Notebookと環境のBinding

Inventory中のgroup名でBind対象ホストを指示する。

Ansible Inventory中に定義されているグループ( `[グループ名]` のように`[]`で囲まれた定義 )のうち、CoursewareHub構築に使うグループ名を指定します。

In [1]:
hosts_file = './hosts'
target_group = 'cwhtest0001'

# 接続確認
%env ANSIBLE_INVENTORY={hosts_file}
!ansible -m ping {target_group}

env: ANSIBLE_INVENTORY=./hosts
10.1.1.157 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}
10.1.1.144 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}
10.1.1.236 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}


NFSサーバーを含むAnsibleのgroup名を指定する。対象が正しいか、Ansible pingモジュールで動作確認する。

In [2]:
target_server = 'cwhub_nfs_server'

!ansible -m ping {target_server} -l {target_group}

10.1.1.236 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}


clientとするnodeはAnsibleのgroup名で定義する。Ansible pingモジュールで動作確認する。

In [3]:
target_nodes = 'cwhub_nodes'

!ansible -m ping {target_nodes} -l {target_group}

10.1.1.144 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}
10.1.1.157 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python"
    },
    "changed": false,
    "ping": "pong"
}


各々のノードには、servicenet_ip変数が定義されている必要がある。

In [4]:
!ansible -m debug -a 'msg={{{{ servicenet_ip }}}}' {target_server} -l {target_group}
!ansible -m debug -a 'msg={{{{ servicenet_ip }}}}' {target_nodes} -l {target_group}

10.1.1.236 | SUCCESS => {
    "msg": "10.1.1.236"
}
10.1.1.144 | SUCCESS => {
    "msg": "10.1.1.144"
}
10.1.1.157 | SUCCESS => {
    "msg": "10.1.1.157"
}


デフォルトユーザを取得しておく。

In [5]:
result = !ansible -a whoami {target_server}
default_user = result[1]
default_user

'centos'

# Playbookの準備

In [6]:
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

'/tmp/tmpty7obgfs'

In [7]:
!rm -fr {work_dir}/jupyterhub-deploy
!git clone https://github.com/NII-cloud-operation/CoursewareHub-LC_jupyterhub-deploy.git {work_dir}/jupyterhub-deploy

Cloning into '/tmp/tmpty7obgfs/jupyterhub-deploy'...
remote: Enumerating objects: 2334, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 2334 (delta 24), reused 67 (delta 23), pack-reused 2257
Receiving objects: 100% (2334/2334), 3.80 MiB | 6.73 MiB/s, done.
Resolving deltas: 100% (995/995), done.


Check the roles...

In [8]:
!ls -la {work_dir}/jupyterhub-deploy/roles

total 4
drwxr-xr-x. 11 niiyazawa users  155 Sep 13 09:00 .
drwxr-xr-x.  7 niiyazawa users 4096 Sep 13 09:00 ..
drwxr-xr-x.  7 niiyazawa users   81 Sep 13 09:00 common
drwxr-xr-x.  5 niiyazawa users   52 Sep 13 09:00 docker
drwxr-xr-x.  6 niiyazawa users   65 Sep 13 09:00 hub
drwxr-xr-x.  4 niiyazawa users   35 Sep 13 09:00 nfs_client
drwxr-xr-x.  5 niiyazawa users   52 Sep 13 09:00 nfs_server
drwxr-xr-x.  5 niiyazawa users   52 Sep 13 09:00 nfs_server_dedicated
drwxr-xr-x.  5 niiyazawa users   52 Sep 13 09:00 proxy
drwxr-xr-x.  5 niiyazawa users   47 Sep 13 09:00 rackspace
drwxr-xr-x.  4 niiyazawa users   32 Sep 13 09:00 restuser


In [9]:
!cd {work_dir}/jupyterhub-deploy/ && git log -n 5

commit 53bd87f4e8b30a0644e7feadc1e67793f0ccfc1e (HEAD -> master, origin/master, origin/HEAD)
Merge: 6f9f9e6 0df1e56
Author: Satoshi Yazawa <yazawa@yzwlab.net>
Date:   Fri Sep 10 06:29:52 2021 +0900

    Merge pull request #4 from yacchin1205/fix/unused-vars
    
    Remove unused variables

commit 0df1e568fefddb759089eec0e798c12c90af4da5
Author: yacchin1205 <yazawa@yzwlab.net>
Date:   Tue Sep 7 13:51:03 2021 +0900

    Remove unused variables

commit 6f9f9e6de29a5ca52f4f12e9f69f4ebd13c4d82d
Merge: c24304e fce81d5
Author: Shingo Nakagawa <shingo.nakagawa@gmail.com>
Date:   Wed Mar 18 20:47:59 2020 +0900

    Merge pull request #3 from shingo78/feature/dedicated_nfs_server
    
    Add nfs_server_dedicated role

commit fce81d5036a8002ebf4238f1aca2ab533207962a
Author: shingo78 <shingo.nakagawa@gmail.com>
Date:   Fri Mar 13 15:31:49 2020 +0900

    Fix bindvolume.yml, remove /mnt/nfs directory

commit 29c515ceaafa28829624cdf2c87c715a569112f3
Author: shingo78 <shingo.nakagawa@gmail.com>
Dat

# NFS serverのインストールと設定

https://github.com/axsh/jupyterhub-deploy/blob/master/deploy-part2.yml#L10-L14

NFSサーバーとCoursewareHubのmasterノードが共用の場合は、以下のAnsible roleを使用する。

`nfs_server_roles = ['nfs_server']`

NFSサーバーとCousewareHubのmasterノードを分離する場合は、以下のAnsible roleを使用する。

`nfs_server_roles = ['nfs_server_dedicated']`

> この切り替えはお試しNotebookでは不要かもしれない

In [10]:
nfs_server_roles = ['nfs_server']
# nfs_server_roles = ['nfs_server_dedicated']

In [11]:
# 適切なroleを設定してから、Freezeして先に進める
assert False

AssertionError: 

In [12]:
import os
import yaml
with open(os.path.join(work_dir, 'jupyterhub-deploy/deploy-nfs-server.yml'), 'w') as f:
    f.write(yaml.dump([{'hosts': target_server,
                       'become': True,
                       'roles': nfs_server_roles}]))
!cat {work_dir}/jupyterhub-deploy/deploy-nfs-server.yml

- become: true
  hosts: cwhub_nfs_server
  roles:
  - nfs_server


Now, dry-run first...

In [13]:
!ansible-playbook -CDv {work_dir}/jupyterhub-deploy/deploy-nfs-server.yml -l {target_group}

Using /etc/ansible/ansible.cfg as config file

PLAY [cwhub_nfs_server] ********************************************************

TASK [Gathering Facts] *********************************************************
ok: [10.1.1.236]

TASK [nfs_server : fail] *******************************************************
skipping: [10.1.1.236] => {"changed": false, "skip_reason": "Conditional result was False"}

TASK [nfs_server : install nfs packages] ***************************************
changed: [10.1.1.236] => {"changed": true, "changes": {"installed": [], "updated": [["nfs-utils", "1:1.3.0-0.68.el7.1.x86_64 from updates"]]}, "msg": "", "obsoletes": {"iwl7265-firmware": {"dist": "noarch", "repo": "installed", "version": "22.0.7.0-72.el7"}}, "rc": 0, "results": []}

TASK [nfs_server : start nfs] **************************************************
changed: [10.1.1.236] => {"changed": true, "enabled": true, "name": "nfs-server", "state": "started", "status": {"ActiveEnterTimestampMonotonic": "0", 

--- before
+++ after
@@ -1,4 +1,4 @@
 {
     "path": "/exported/test01",
-    "state": "absent"
+    "state": "directory"
 }

changed: [10.1.1.236] => {"changed": true, "path": "/exported/test01"}

TASK [nfs_server : create sub directory for exchange volume] *******************
--- before
+++ after
@@ -1,4 +1,4 @@
 {
     "path": "/exported/test01/exchange",
-    "state": "absent"
+    "state": "directory"
 }

changed: [10.1.1.236] => {"changed": true, "path": "/exported/test01/exchange"}

TASK [nfs_server : create sub directory for jupyter volume] ********************
--- before
+++ after
@@ -1,4 +1,4 @@
 {
     "path": "/exported/test01/jupyter",
-    "state": "absent"
+    "state": "directory"
 }

changed: [10.1.1.236] => {"changed": true, "path": "/exported/test01/jupyter"}

TASK [nfs_server : modify /etc/hosts.allow] ************************************
--- before: /etc/hosts.allow (content)
+++ after: /etc/hosts.allow (content)
@@ -8,3 +8,4 @@
 #		for information on rule syntax.


It seems OK, apply it to hosts.

In [14]:
!ansible-playbook {work_dir}/jupyterhub-deploy/deploy-nfs-server.yml -l {target_group}


PLAY [cwhub_nfs_server] ********************************************************

TASK [Gathering Facts] *********************************************************
ok: [10.1.1.236]

TASK [nfs_server : fail] *******************************************************
skipping: [10.1.1.236]

TASK [nfs_server : install nfs packages] ***************************************
changed: [10.1.1.236]

TASK [nfs_server : start nfs] **************************************************
changed: [10.1.1.236]

TASK [nfs_server : start idmapd] ***********************************************
ok: [10.1.1.236]

TASK [nfs_server : create root directory for NFS volume] ***********************
changed: [10.1.1.236]

TASK [nfs_server : create sub directory for exchange volume] *******************
changed: [10.1.1.236]

TASK [nfs_server : create sub directory for jupyter volume] ********************
changed: [10.1.1.236]

TASK [nfs_server : modify /etc/hosts.allow] ************************************
changed: [10.

# NFS clientのインストールと設定

In [15]:
with open(os.path.join(work_dir, 'jupyterhub-deploy/deploy-nfs-client.yml'), 'w') as f:
    f.write(yaml.dump([{'hosts': target_nodes,
                       'become': True,
                       'roles': ['nfs_client']}]))
!cat {work_dir}/jupyterhub-deploy/deploy-nfs-client.yml

- become: true
  hosts: cwhub_nodes
  roles:
  - nfs_client


In [16]:
!ansible-playbook -CDv {work_dir}/jupyterhub-deploy/deploy-nfs-client.yml -l {target_group}

Using /etc/ansible/ansible.cfg as config file

PLAY [cwhub_nodes] *************************************************************

TASK [Gathering Facts] *********************************************************
ok: [10.1.1.157]
ok: [10.1.1.144]

TASK [nfs_client : fail] *******************************************************
skipping: [10.1.1.144] => {"changed": false, "skip_reason": "Conditional result was False"}
skipping: [10.1.1.157] => {"changed": false, "skip_reason": "Conditional result was False"}

TASK [nfs_client : fail] *******************************************************
skipping: [10.1.1.144] => {"changed": false, "skip_reason": "Conditional result was False"}
skipping: [10.1.1.157] => {"changed": false, "skip_reason": "Conditional result was False"}

TASK [nfs_client : install nfs client] *****************************************
ok: [10.1.1.157] => {"changed": false, "msg": "", "rc": 0, "results": ["1:nfs-utils-1.3.0-0.65.el7.x86_64 providing nfs-utils is already insta

OK, apply it.

In [17]:
!ansible-playbook {work_dir}/jupyterhub-deploy/deploy-nfs-client.yml -l {target_group}


PLAY [cwhub_nodes] *************************************************************

TASK [Gathering Facts] *********************************************************
ok: [10.1.1.144]
ok: [10.1.1.157]

TASK [nfs_client : fail] *******************************************************
skipping: [10.1.1.144]
skipping: [10.1.1.157]

TASK [nfs_client : fail] *******************************************************
skipping: [10.1.1.144]
skipping: [10.1.1.157]

TASK [nfs_client : install nfs client] *****************************************
ok: [10.1.1.144]
ok: [10.1.1.157]

TASK [nfs_client : unmount exchange nsf dir] ***********************************
changed: [10.1.1.144]
changed: [10.1.1.157]

TASK [nfs_client : unmount jupyter nsf dir] ************************************
changed: [10.1.1.144]
changed: [10.1.1.157]

TASK [nfs_client : mount exchange nfs dir] *************************************
changed: [10.1.1.157]
changed: [10.1.1.144]

TASK [nfs_client : mount jupyter nfs dir] *********

# NFSの動作確認


In [18]:
!ansible -a 'df -h' {target_nodes} -l {target_group}

10.1.1.157 | CHANGED | rc=0 >>
Filesystem            Size  Used Avail Use% Mounted on
devtmpfs              1.9G     0  1.9G   0% /dev
tmpfs                 1.9G     0  1.9G   0% /dev/shm
tmpfs                 1.9G   17M  1.9G   1% /run
tmpfs                 1.9G     0  1.9G   0% /sys/fs/cgroup
/dev/xvda1             50G  918M   50G   2% /
tmpfs                 379M     0  379M   0% /run/user/0
tmpfs                 379M     0  379M   0% /run/user/1000
10.1.1.236:/exchange  100G  1.1G   99G   2% /exchange
10.1.1.236:/jupyter   100G  1.1G   99G   2% /jupyter
10.1.1.144 | CHANGED | rc=0 >>
Filesystem            Size  Used Avail Use% Mounted on
devtmpfs              1.9G     0  1.9G   0% /dev
tmpfs                 1.9G     0  1.9G   0% /dev/shm
tmpfs                 1.9G   17M  1.9G   1% /run
tmpfs                 1.9G     0  1.9G   0% /sys/fs/cgroup
/dev/xvda1             50G  918M   50G   2% /
tmpfs                 379M     0  379M   0% /run/user/0
tmpfs                 379M     0  379M

ファイルが見えるか、確認する。

In [19]:
!ansible -a 'ls -la {{{{ nfspath }}}}/exchange' {target_server} -l {target_group}
!ansible -a 'ls -la /exchange' {target_nodes} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
total 0
drwxrwxrwx. 2 root root  6 Sep 13 09:01 .
drwxr-xr-x. 4 root root 37 Sep 13 09:01 ..
10.1.1.144 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root   6 Sep 13 09:01 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..
10.1.1.157 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root   6 Sep 13 09:01 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..


## 動作確認その1

Create a test file on `/exchange`...

In [20]:
!ansible -b -a 'touch {{{{ nfspath }}}}/exchange/test' {target_server} -l {target_group}

[WARNING]: Consider using the file module with state=touch rather than running
'touch'.  If you need to use command because file is insufficient you can add
'warn: false' to this command task or set 'command_warnings=False' in
ansible.cfg to get rid of this message.
10.1.1.236 | CHANGED | rc=0 >>



Check whether `exchange` directory of hub and nodes are sync.

In [21]:
!ansible -a 'ls -la {{{{ nfspath }}}}/exchange' {target_server} -l {target_group}
!ansible -a 'ls -la /exchange' {target_nodes} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
total 0
drwxrwxrwx. 2 root root 18 Sep 13 09:02 .
drwxr-xr-x. 4 root root 37 Sep 13 09:01 ..
-rw-r--r--. 1 root root  0 Sep 13 09:02 test
10.1.1.144 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root  18 Sep 13 09:02 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..
-rw-r--r--.  1 root root   0 Sep 13 09:02 test
10.1.1.157 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root  18 Sep 13 09:02 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..
-rw-r--r--.  1 root root   0 Sep 13 09:02 test


OK, the test file also appears in NFS clients.

Remove the test file.

In [22]:
!ansible -b -a 'rm {{{{ nfspath }}}}/exchange/test' {target_server} -l {target_group}

[WARNING]: Consider using the file module with state=absent rather than running
'rm'.  If you need to use command because file is insufficient you can add
'warn: false' to this command task or set 'command_warnings=False' in
ansible.cfg to get rid of this message.
10.1.1.236 | CHANGED | rc=0 >>



In [23]:
!ansible -a 'ls -la {{{{ nfspath }}}}/exchange' {target_server} -l {target_group}
!ansible -a 'ls -la /exchange' {target_nodes} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
total 0
drwxrwxrwx. 2 root root  6 Sep 13 09:03 .
drwxr-xr-x. 4 root root 37 Sep 13 09:01 ..
10.1.1.144 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root   6 Sep 13 09:03 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..
10.1.1.157 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root   6 Sep 13 09:03 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..


## 再起動しても動く？

Reboot VM... NFS clientだけ。

NFS serverは再起動しない

In [24]:
!ansible -b -m reboot {target_nodes} -l {target_group}

10.1.1.144 | CHANGED => {
    "changed": true,
    "elapsed": 78,
    "rebooted": true
}
10.1.1.157 | CHANGED => {
    "changed": true,
    "elapsed": 105,
    "rebooted": true
}


## 動作確認その2

再起動後でも動くか？

In [25]:
!ansible -b -a 'touch {{{{ nfspath }}}}/exchange/test' {target_server} -l {target_group}

[WARNING]: Consider using the file module with state=touch rather than running
'touch'.  If you need to use command because file is insufficient you can add
'warn: false' to this command task or set 'command_warnings=False' in
ansible.cfg to get rid of this message.
10.1.1.236 | CHANGED | rc=0 >>



In [26]:
!ansible -a 'ls -la {{{{ nfspath }}}}/exchange' {target_server} -l {target_group}
!ansible -a 'ls -la /exchange' {target_nodes} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
total 0
drwxrwxrwx. 2 root root 18 Sep 13 09:05 .
drwxr-xr-x. 4 root root 37 Sep 13 09:01 ..
-rw-r--r--. 1 root root  0 Sep 13 09:05 test
10.1.1.144 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root  18 Sep 13 09:05 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..
-rw-r--r--.  1 root root   0 Sep 13 09:05 test
10.1.1.157 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root  18 Sep 13 09:05 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..
-rw-r--r--.  1 root root   0 Sep 13 09:05 test


In [27]:
!ansible -b -a 'rm {{{{ nfspath }}}}/exchange/test' {target_server} -l {target_group}

[WARNING]: Consider using the file module with state=absent rather than running
'rm'.  If you need to use command because file is insufficient you can add
'warn: false' to this command task or set 'command_warnings=False' in
ansible.cfg to get rid of this message.
10.1.1.236 | CHANGED | rc=0 >>



In [28]:
!ansible -a 'ls -la {{{{ nfspath }}}}/exchange' {target_server} -l {target_group}
!ansible -a 'ls -la /exchange' {target_nodes} -l {target_group}

10.1.1.236 | CHANGED | rc=0 >>
total 0
drwxrwxrwx. 2 root root  6 Sep 13 09:05 .
drwxr-xr-x. 4 root root 37 Sep 13 09:01 ..
10.1.1.144 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root   6 Sep 13 09:05 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..
10.1.1.157 | CHANGED | rc=0 >>
total 0
drwxrwxrwx.  2 root root   6 Sep 13 09:05 .
dr-xr-xr-x. 19 root root 255 Sep 13 09:02 ..


# 後始末

In [29]:
!rm -rf {work_dir}